# Data Pre-processing

## Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv('dataset/tickets.csv')

df_clean = df[['DESKRIPSI']].copy()

df_clean = df_clean.dropna()

print(f"Total data: {len(df_clean)}")
df_clean.head()

Total data: 1639


,DESKRIPSI
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...


## Normalisasi

In [2]:
def normalize_text(text):
    # Mengubah ke lower case
    return text.lower()

df_clean['NORMAL_TEXT'] = df_clean['DESKRIPSI'].apply(normalize_text)
df_clean[['DESKRIPSI', 'NORMAL_TEXT']].head()

,DESKRIPSI,NORMAL_TEXT
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,klien: setwapres medsos\nisu: crawlback commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,klien: bps\nisu: data postingan instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,klien: heinz \nisu: dashboard loading\n\nselam...


## Cleansing

In [3]:
import re
import unicodedata

def cleansing_text(text):    
    # Penghapusan URL dan akun media sosial (misalnya, @username, #hashtag, user_name, user.name)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\S*_\S*|\S*\.\S*', ' ', text)

    # Menghapus nama platform media sosial yang umum (instagram, facebook, tiktok, dll)
    sosmed = ['facebook', 'twitter', 'instagram', 'linkedin', 'youtube', 'tiktok', 'snapchat', 'pinterest']
    pattern_sosmed = r'\w*(?:' + '|'.join(sosmed) + r')\w*'
    text = re.sub(pattern_sosmed, ' ', text)

    # Menghapus bulan (baik yang lengkap maupun yang disingkat)
    bulan = ['januari', 'februari', 'maret', 'april', 'mei', 'juni', 
             'juli', 'agustus', 'september', 'oktober', 'november', 'desember',
             'jan', 'feb', 'mar', 'apr', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
    # Regex untuk menghapus kata bulan yang berdiri sendiri
    pattern_bulan = r'\b(' + '|'.join(bulan) + r')\b'
    text = re.sub(pattern_bulan, ' ', text)
    
    # Menggabungkan baris (menangani enter yang tersembunyi)
    text = ' '.join(text.splitlines())
    
    # Menghapus karakter khusus/simbol (hanya menyisakan huruf dan angka)
    text = re.sub(r'[^\w\s]', ' ', text)

    # Menghapus deretan angka (seperti tanggal, jam, nomor tiket)
    text = re.sub(r'\d+', ' ', text)
    
    # Meringkas segala jenis whitespace (spasi, tab, newline) menjadi SATU spasi
    # Regex \s+ akan mencari satu atau lebih karakter whitespace berturut-turut
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

df_clean['CLEAN_TEXT'] = df_clean['NORMAL_TEXT'].apply(cleansing_text)
df_clean[['NORMAL_TEXT', 'CLEAN_TEXT']].head()

,NORMAL_TEXT,CLEAN_TEXT
0,klien: setwapres medsos\nisu: crawlback commen...,klien setwapres medsos isu crawlback comment s...
1,"selamat pagi tim it, mohon bantuannya saya men...",selamat pagi tim it mohon bantuannya saya mene...
2,klien: bps\nisu: data postingan instagram tida...,klien bps isu data postingan tidak masuk selam...
3,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas dan tim info untuk siputri ti...
4,klien: heinz \nisu: dashboard loading\n\nselam...,klien heinz isu dashboard loading selamat sore...


## Save Preprocessed Data

In [5]:
output_file = 'dataset/tickets_preprocessed.csv'

df_clean[['CLEAN_TEXT']].to_csv(output_file, index=False)

print(f"Data berhasil disimpan ke dalam file: {output_file}")

Data berhasil disimpan ke dalam file: dataset/tickets_preprocessed.csv
